In [1]:
import os

# Pick a GPU BEFORE importing torch/swift.
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3'
# Uncomment to see the 3D encoder / token-routing debug logs during the forward pass:
# os.environ['VISION_3D_DEBUG'] = '1'

# Make the local swift fork importable (this notebook lives in vlm/swift/notebooks).
import sys
sys.path.insert(0, os.path.abspath('..'))

from swift.arguments import InferArguments
from swift.pipelines.utils import prepare_model_template
from swift.infer_engine import TransformersEngine, InferRequest, RequestConfig
from swift.metrics import InferStats

[INFO:swift] Conv3d patched successfully
[INFO:swift] Successfully registered `/cache/fast_data_nas71/janhavi/VLM_qure/vlm/swift/swift/dataset/data/dataset_info.json`.


In [2]:
# ---- things you'll change ---------------------------------------------------
CKPT = '/cache/fast_data_nas71/janhavi/VLM_qure/vlm/swift/vlm_ckpts/2907_qwen3_5_4b_atlas_3d_ct/v6-20260804-010105/checkpoint-26000'

# System prompt (the checkpoint was trained with this one; leave as-is unless you want to override).
SYSTEM = 'You are a helpful medical assistant.'

In [3]:
# ---- load the checkpoint ----------------------------------------------------
# InferArguments(model=CKPT) auto-reads the checkpoint's args.json (restores template
# 'qwen2_5_vl_ct', system, torch_dtype, ...). prepare_model_template then:
#   * builds the model, and
#   * attaches the trained 3D CT encoder as model.visual_3d (rebuilt from config.json's
#     vision_3d_config + the trained visual_3d.* weights), and
#   * builds the CT template with the right ct_windows / ct_volume_size / vision_3d_max_tokens.
# The checkpoint was trained in float32; pass torch_dtype='bfloat16' below if you hit OOM.
args = InferArguments(
    model=CKPT,
    # torch_dtype='bfloat16',   # uncomment to halve memory
    max_new_tokens=4096,
)

model, template = prepare_model_template(args)
engine = TransformersEngine(model, template=template, max_batch_size=1)
print('loaded. visual_3d attached:', hasattr(model, 'visual_3d'))

[INFO:swift] Successfully loaded /cache/fast_data_nas71/janhavi/VLM_qure/vlm/swift/vlm_ckpts/2907_qwen3_5_4b_atlas_3d_ct/v6-20260804-010105/checkpoint-26000/args.json.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] Setting args.eval_human: True
[INFO:swift] Setting args.lazy_tokenize: True
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] attn_impl: sdpa
[INFO:swift] Setting max_ratio: 200. You can adjust this hyperparameter through the environment variable: `MAX_RATIO`.
[INFO:swift] Setting frame_factor: 2. You can adjust this hyperparameter through the environment variable: `FRAME_FACTOR`.
[INFO:swift] Setting fps: 2.0. You can adjust this hyperparameter through the environment variable: `FPS`.
[INFO:swift] Setting fps_min_frames: 4. You can adjust this hyperparameter through t

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] Qwen3_5ForConditionalGeneration LOAD REPORT from: /cache/fast_data_nas71/janhavi/VLM_qure/vlm/swift/vlm_ckpts/2907_qwen3_5_4b_atlas_3d_ct/v6-20260804-010105/checkpoint-26000
Key                                                                                                             | Status     |  | 
----------------------------------------------------------------------------------------------------------------+------------+--+-
visual_3d.encoder.atlas_models.{0, 1, 2}.blocks.{0, 1, 2, 3, 4, 5, 6, 7}.blocks.{0, 1, 2}.mlp.fc2.weight        | UNEXPECTED |  | 
visual_3d.encoder.atlas_models.{0, 1, 2}.blocks.{0, 1, 2, 3, 4, 5, 6, 7}.blocks.{0, 1, 2}.attn.q.weight         | UNEXPECTED |  | 
visual_3d.encoder.atlas_models.{0, 1, 2}.blocks.{0, 1, 2, 3, 4, 5, 6, 7}.blocks.{0, 1, 2}.mlp.fc1.weight        | UNEXPECTED |  | 
visual_3d.encoder.atlas_models.{0, 1, 2}.blocks.{0, 1, 2, 3, 4, 5, 6, 7}.posemb.{0, 1, 2}.cpb_mlp.{0, 2}.weight | UNEXPECTED |  | 
visual_3d.encoder.atlas_m

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

[INFO:swift] vision_3d: detected Atlas (clip_multimodal_atlas); building vision tower only.
/cache/fast_data_nas71/janhavi/miniconda3/envs/swift_infer/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[INFO:swift] vision_3d/atlas: built MultiModalAtlas vision tower and loaded 747 tensors (missing=0, unexpected=0).
[INFO:swift] vision_3d: loaded model is itself a vision tower (MultiModalAtlas); using as-is.
[INFO:swift] vision_3d: patch embed `patch_embeds.chest_ct.conv_down.0` is already nn.Conv3d with in_channels=11; no change needed.
[INFO:swift] vision_3d/atlas: set encoder image_size=[256, 256, 256] to match --ct_volume_size (must be large enough to yield the checkpoint's scale count, e.g. >=128^3).
[INFO:swift] vision_3d: projector 384 -> 2560 (in_channels=11, max_tokens

loaded. visual_3d attached: True


In [4]:
# ---- scroll through a CT volume, slice by slice -----------------------------
# Uses the same loader the model uses (read_hu_volume): handles local paths, s3://rnd-data-lake
# URIs, .safetensors / .nii(.gz) / .npy / DICOM-dir. Displays raw HU with a window/level control.
import numpy as np
import matplotlib.pyplot as plt
from swift.utils.ct_volume_io import read_hu_volume

# Common CT window presets: (width, level) in HU.
CT_WINDOWS = {
    'lung': (1500, -600), 'mediastinum': (350, 40), 'abdomen': (400, 40),
    'bone': (1800, 400), 'brain': (80, 40), 'full_range': (2000, 0),
}

_vol_cache = {}
def _get_volume(path):
    if path not in _vol_cache:
        v = read_hu_volume(path).numpy()  # (D, H, W) in HU
        _vol_cache[path] = v
        print(f'loaded {path}\n  shape (D,H,W)={v.shape}  HU range [{v.min():.0f}, {v.max():.0f}]')
    return _vol_cache[path]

def plot_ct_volume(ct_volume_path=None, axis=0, window='lung'):
    """Interactive slice viewer. axis: 0=axial, 1=coronal, 2=sagittal.

    Drag the `idx` slider to scroll; `width`/`level` adjust the HU window (start from a preset above).
    """
    ct_volume_path = ct_volume_path or CT_VOLUME_PATH
    vol = _get_volume(ct_volume_path)
    n = vol.shape[axis]
    w0, l0 = CT_WINDOWS.get(window, CT_WINDOWS['lung'])

    def show(idx, width, level):
        sl = np.take(vol, idx, axis=axis)
        plt.figure(figsize=(6, 6))
        plt.imshow(sl, cmap='gray', vmin=level - width / 2, vmax=level + width / 2)
        plt.title(f'{["axial", "coronal", "sagittal"][axis]}  slice {idx}/{n - 1}   W:{width} L:{level}')
        plt.axis('off')
        plt.show()

    try:
        from ipywidgets import interact, IntSlider
        interact(show,
                 idx=IntSlider(min=0, max=n - 1, step=1, value=n // 2, continuous_update=False),
                 width=IntSlider(min=1, max=3000, step=10, value=w0, continuous_update=False),
                 level=IntSlider(min=-1024, max=1024, step=10, value=l0, continuous_update=False))
    except Exception as e:
        # Fallback if ipywidgets isn't working: a static montage of evenly-spaced slices.
        print(f'ipywidgets unavailable ({e}); showing a static montage instead.')
        idxs = np.linspace(0, n - 1, 16).astype(int)
        fig, axes = plt.subplots(4, 4, figsize=(12, 12))
        for ax, i in zip(axes.ravel(), idxs):
            ax.imshow(np.take(vol, i, axis=axis), cmap='gray', vmin=l0 - w0 / 2, vmax=l0 + w0 / 2)
            ax.set_title(f'{i}', fontsize=8)
            ax.axis('off')
        plt.tight_layout(); plt.show()

In [5]:
# The CT volume. May be a local path or an s3://rnd-data-lake/... URI.
# Supported: .safetensors (loaded during training), .nii/.nii.gz, .npy/.npz, or a DICOM dir.
# The volume rides the model's <video> channel and is routed to model.visual_3d.
CT_VOLUME_PATH = 's3://rnd-data-lake/safetensors/1.3.6.1.4.1.55648.2.462672134531155705212156357760752918265.safetensors'

# The three task prompts the training datasets were built with (one per dataset). Uncomment the
# one to try; the <video> placeholder is appended automatically at inference time.
PROMPT = 'Generate the report for the given CT volume.'                                                                                       # 15_July autoreporting
# PROMPT = 'Classify the abnormalities in the given CT volume in JSON format with presence, size, location and very short characterization.'  # 21_July json tag extraction
# PROMPT = 'Classify the abnormalities from the given CT volume.'                                                                             # 21_July cls-token tag extraction

In [6]:
# ---- build the messages + run inference -------------------------------------
# The CT volume goes in the `videos` field; the `<video>` tag in the text marks where it
# is spliced in (mirrors the training data: "Generate the report ... <video>"). The number
# of <video> tags must equal len(videos).
messages = [
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': f'{PROMPT} <video>'},
]

infer_request = InferRequest(messages=messages, videos=[CT_VOLUME_PATH])

request_config = RequestConfig(max_tokens=4096, temperature=0.4)
metric = InferStats()
resp = engine.infer([infer_request], request_config, metrics=[metric])

print('PROMPT :', PROMPT)
print('VOLUME :', CT_VOLUME_PATH)
print('=' * 80)
print(resp[0].choices[0].message.content)
print('=' * 80)
print(metric.compute())

PROMPT : Generate the report for the given CT volume.
VOLUME : s3://rnd-data-lake/safetensors/1.3.6.1.4.1.55648.2.462672134531155705212156357760752918265.safetensors
 with reconstructions performed in axial, sagittal and coronal planes. This study was performed using automatic exposure control and an iterative reconstruction technique (radiation dose reduction software) to obtain a diagnostic image quality scan with patient dose as low as reasonably achievable. mA and kV were adjusted according to patient's size. The administered radiation dose was 2.03 mSv. Comparison is made with chest CT from segmed_PHI/DD/YYYY and MM/DD/YYYY. THORACIC INLET: No suspicious mass. LUNGS AND PLEURA: There is no significant change in known 5 mm nodule in the right middle lobe on image 49 of series 3. There is no suspicious new lesion. segmed_LASTNAME, segmed_FIRSTNAME segmed_MIDDLENAME AXILLAE: No suspicious mass or adenopathy. HEART AND VESSELS: Heart is not enlarged. There is no aneurysm. CHEST WALL/S

In [7]:
plot_ct_volume(CT_VOLUME_PATH, axis=0, window='lung')

loaded s3://rnd-data-lake/safetensors/1.3.6.1.4.1.55648.2.462672134531155705212156357760752918265.safetensors
  shape (D,H,W)=(105, 512, 512)  HU range [-1024, 1828]


interactive(children=(IntSlider(value=52, continuous_update=False, description='idx', max=104), IntSlider(valu…

In [8]:
# ---- convenience: re-run with a new prompt/volume without reloading the model ----
def run(prompt, ct_volume_path, system=SYSTEM, max_tokens=1024, temperature=0.0,
        stream=False, verbose=True):
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': f'{prompt} <video>'},
    ]
    req = InferRequest(messages=messages, videos=[ct_volume_path])
    cfg = RequestConfig(max_tokens=max_tokens, temperature=temperature, stream=stream)
    if stream:
        text = ''
        for chunk in engine.infer([req], cfg)[0]:
            delta = chunk.choices[0].delta.content
            print(delta, end='', flush=True)
            text += delta
        print()
        return text
    out = engine.infer([req], cfg)[0].choices[0].message.content
    if verbose:
        print(out)
    return out

# e.g.:
# run('What are the key findings in the lungs?', CT_VOLUME_PATH, stream=True)

In [9]:
# ---- load the CT-RATE eval parquet ------------------------------------------
# CT-RATE eval set built from rnd-data-lake: one row per CT series, with a verified
# safetensors volume (`volume_uri`) AND its ground-truth radiology report (`report`).
# Columns: SeriesInstanceUID, StudyInstanceUID, PatientID, num_slices, volume_uri, report.
# Guaranteed disjoint from the liger CT training data, so it's a clean eval set.
import pandas as pd
import textwrap

PARQUET_PATH = '/cache/fast_data_nas71/janhavi/data/ct_eval/ct_rate_eval.parquet'

_df = pd.read_parquet(PARQUET_PATH)
_df = _df[_df['report'].notna()].reset_index(drop=True)
print(f'{len(_df)} CT-RATE rows (series) with reports in {PARQUET_PATH}')
print(f"  studies={_df['StudyInstanceUID'].nunique()}  patients={_df['PatientID'].nunique()}")

def get_sample(i=None, seed=None, verbose=True):
    """Pick a parquet row (random if i is None) -> dict with volume_path + ground-truth report.

    Uses the row's `volume_uri` directly (CT-RATE volumes use native names like
    `train_11223_a_2`, not raw SeriesInstanceUIDs). Sets the notebook-global CT_VOLUME_PATH
    so plot_ct_volume()/the inference cell pick it up. Set verbose=False to skip the printout
    (compare() does this to avoid printing the report twice).
    """
    global CT_VOLUME_PATH
    if i is None:
        i = int(_df.sample(1, random_state=seed).index[0])
    row = _df.iloc[i]
    sample = {
        'index': i,
        'series_uid': row['SeriesInstanceUID'],
        'volume_path': row['volume_uri'],
        'num_slices': int(row['num_slices']),
        'report': str(row['report']),
    }
    CT_VOLUME_PATH = sample['volume_path']
    if verbose:
        print(f"[row {i}] series_uid: {sample['series_uid']}  ({sample['num_slices']} slices)")
        print(f"volume_path: {sample['volume_path']}")
        print('-' * 80, '\nGROUND TRUTH REPORT:\n')
        print('\n'.join(textwrap.fill(line, 100) for line in sample['report'].splitlines()))
    return sample

# --- typical workflow (see the compare cell below) ---------------------------
# s = get_sample()                    # random row (or get_sample(1234) for a specific one)
# plot_ct_volume(s['volume_path'])    # scroll through the volume
# out = run(PROMPT, s['volume_path']) # model prediction -> compare against s['report']
#
# Note: every CT-RATE row here has a verified volume in the bucket, but the first access
# still downloads it into the local cache (can take a few seconds).

50010 CT-RATE rows (series) with reports in /cache/fast_data_nas71/janhavi/data/ct_eval/ct_rate_eval.parquet
  studies=25598  patients=21245


In [13]:
# ---- sample a random CT-RATE case -> view volume -> compare model vs GT ------
# Re-run this cell to draw a new random case. It:
#   1) samples a random row (volume + ground-truth report),
#   2) opens the interactive slice viewer (drag `idx` to scroll; `width`/`level` = HU window),
#   3) runs the model on the volume and prints the prediction next to the GT report.
import textwrap

def compare(i=None, seed=None, prompt=None, window='lung', axis=0,
            max_tokens=4096, temperature=0.0):
    prompt = prompt or PROMPT
    sample = get_sample(i=i, seed=seed, verbose=False)   # sets CT_VOLUME_PATH (printout below)

    # interactive slice viewer (slider). Fallback to a static montage if ipywidgets is off.
    plot_ct_volume(sample['volume_path'], axis=axis, window=window)

    # model prediction (verbose=False: we lay out pred + GT ourselves below)
    pred = run(prompt, sample['volume_path'], max_tokens=max_tokens,
               temperature=temperature, verbose=False)

    wrap = lambda t: '\n'.join(textwrap.fill(l, 100) for l in str(t).splitlines())
    print('\n' + '=' * 100)
    print(f"CASE row {sample['index']}  |  series {sample['series_uid']}  |  {sample['num_slices']} slices")
    print(f"PROMPT: {prompt}")
    print('=' * 100)
    print('\n########## MODEL PREDICTION ##########\n')
    print(wrap(pred))
    print('\n########## GROUND-TRUTH REPORT ##########\n')
    print(wrap(sample['report']))
    print('=' * 100)
    return {**sample, 'prediction': pred}

# random case (pass an index for a specific row, e.g. compare(1234), or compare(seed=0) to fix it):
result = compare()

loaded s3://rnd-data-lake/safetensors/train_7965_a_2.safetensors
  shape (D,H,W)=(282, 512, 512)  HU range [-8192, 3096]


interactive(children=(IntSlider(value=141, continuous_update=False, description='idx', max=281), IntSlider(val…


CASE row 40635  |  series train_7965_a_2  |  282 slices
PROMPT: Generate the report for the given CT volume.

########## MODEL PREDICTION ##########

 Shortness Of Breath          R06.2 Wheezing          R06.00
Dyspnea, unspecified         R53.83 Fatigue           R07.9 Chest Pain
R07.1 Chest Pain On Breathing          R07.81 Pleurodynia         J45.998 Other
asthma

TECHNIQUE: Noncontrast CT of the chest was performed. Axial, coronal and
sagittal images were reconstructed using iterative reconstruction technique.
This study was performed using automatic exposure control (radiation dose
reduction software) to obtain a diagnostic image quality scan with patient dose
as low as reasonably achievable.  mA and kV were adjusted according to patient
size. The administered radiation dose was 3.1 mSv.

The patient provides a never before smoking history.
Note is made a history of segmed_LOC exposure. The patient notes a
history of reflux and chest pain

COMPARISON:  Note is made of the chest r